# Model Evaluation: Autoregressive Vorticity Rollout

Visualizes output from `ml evaluate autoregression`. Loads a single NC file produced
by that command and shows:

| # | Visualization | What it shows |
|---|---|---|
| 1 | Side-by-side rotating globes | `vor` (truth) vs `vor_pred` (model), synchronized rotation |
| 2 | Relative error map | `vor_pred_err / vlim_vor` per timestep on orthographic globe |

**Point `NC_FILE` at any eval output to re-run all cells.**

In [ ]:
# --- Parameters: edit these to point at a different eval output ---

NC_FILE = "/tmp/eval.nc"  # output of: ml evaluate autoregression ... --output <path>

# Globe animation
N_FRAMES = 72       # number of animation frames
FPS = 8             # frames per second
ROTATIONS = 1.0     # full rotations over the animation
CENTER_LAT = 30.0   # viewing latitude
START_LON = 0.0     # starting longitude
DPI = 100

# Relative error colormap clamp (fraction of typical vorticity magnitude)
ERR_CLAMP = 0.5     # saturates at 50% of vlim_vor; increase if error is large

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import cartopy.crs as ccrs
import xarray as xr

plt.rcParams["figure.dpi"] = 100

In [ ]:
ds = xr.open_dataset(NC_FILE, decode_times=False)

# Fix ISCA calendar metadata so xarray doesn't choke on it
for v in ds.variables:
    if "units" in ds[v].attrs and "0000-00-00" in ds[v].attrs["units"]:
        ds[v].attrs["units"] = ds[v].attrs["units"].replace("0000-00-00", "0001-01-01")
    if ds[v].attrs.get("calendar") == "NO_CALENDAR":
        ds[v].attrs["calendar"] = "360_day"

lon = ds.lon.values
lat = ds.lat.values
ntime = ds.sizes["time"]

vor      = ds["vor"].values          # (T+1, lat, lon)  ground truth
vor_pred = ds["vor_pred"].values      # (T+1, lat, lon)  model prediction
vor_err  = ds["vor_pred_err"].values  # (T+1, lat, lon)  signed error (pred - truth)

# Shared color limit: 98th percentile of ground-truth vorticity magnitude
vlim_vor = float(np.percentile(np.abs(vor), 98))

print(f"Loaded: {NC_FILE}")
print(f"Time steps: {ntime}  Grid: {len(lat)} x {len(lon)}")
print(f"vlim_vor = {vlim_vor:.4e} s^-1")
print(f"RMS error: {np.sqrt(np.mean(vor_err**2)):.4e} s^-1")

---
## 1. Side-by-Side Rotating Globes: `vor` vs `vor_pred`

Both globes share the same color scale and rotate in sync. Timestep advances
linearly from `t=0` to `t=T` over the animation.

**What to look for:**
- Large-scale structure: does the model capture the eddy positions and signs?
- Fine-scale differences: where does the prediction drift from truth?
- Growth of discrepancies over time

In [ ]:
pc = ccrs.PlateCarree()
rotation_per_frame = (ROTATIONS * 360.0) / N_FRAMES
time_indices = np.linspace(0, ntime - 1, N_FRAMES).astype(int)

fig = plt.figure(figsize=(12, 5))

def make_frame_globes(i):
    fig.clf()
    central_lon = (START_LON + i * rotation_per_frame) % 360
    proj = ccrs.Orthographic(central_lon, CENTER_LAT)
    t = time_indices[i]

    for col, (title, data) in enumerate([
        (f"vor (truth, t={t})", vor[t]),
        (f"vor_pred (model, t={t})", vor_pred[t]),
    ]):
        ax = fig.add_subplot(1, 2, col + 1, projection=proj)
        im = ax.pcolormesh(
            lon, lat, data,
            cmap="RdBu_r", vmin=-vlim_vor, vmax=vlim_vor,
            shading="auto", transform=pc,
        )
        ax.set_global()
        ax.gridlines(alpha=0.3)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, label="vor [s^-1]", shrink=0.7, pad=0.05)

    fig.suptitle("Vorticity: Truth vs Model Prediction", fontsize=12)
    fig.tight_layout()
    return []

ani_globes = animation.FuncAnimation(fig, make_frame_globes, frames=N_FRAMES, blit=False)
plt.close()
HTML(ani_globes.to_jshtml(fps=FPS))

---
## 2. Relative Error Map

Signed relative error at each timestep: $\hat{\zeta}_{\text{err}} / \zeta_{\text{lim}}$
where $\zeta_{\text{lim}}$ is the 98th percentile of $|\zeta_{\text{truth}}|$.

Red = model over-predicts vorticity. Blue = model under-predicts.

**What to look for:**
- Error growing with time (error accumulation in autoregression)
- Systematic spatial bias: does the model consistently err in certain regions?
- Error concentrated at eddy boundaries vs. eddy centers

In [ ]:
rel_err = vor_err / vlim_vor  # dimensionless, in [-ERR_CLAMP, ERR_CLAMP] range

ncols = min(ntime, 4)
nrows = int(np.ceil(ntime / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(5 * ncols, 4.5 * nrows),
    subplot_kw={"projection": ccrs.Orthographic(START_LON, CENTER_LAT)},
)
axes_flat = np.array(axes).flat

for t in range(ntime):
    ax = axes_flat[t]
    im = ax.pcolormesh(
        lon, lat, rel_err[t],
        cmap="RdBu_r",
        vmin=-ERR_CLAMP, vmax=ERR_CLAMP,
        shading="auto", transform=pc,
    )
    ax.set_global()
    ax.gridlines(alpha=0.3)
    ax.set_title(f"t={t}")
    fig.colorbar(im, ax=ax, label="rel. error", shrink=0.7, pad=0.05)

for j in range(ntime, nrows * ncols):
    axes_flat[j].set_visible(False)

fig.suptitle(
    f"Relative Vorticity Error  (vor_pred_err / vlim_vor,  clamp={ERR_CLAMP})",
    fontsize=13,
)
plt.tight_layout()
plt.show()

---
## 3. Animated Relative Error on Globe

Same relative error as section 2, but as a rotating globe animation.
Timestep advances with the rotation.

In [ ]:
fig = plt.figure(figsize=(6, 5))

def make_frame_err(i):
    fig.clf()
    central_lon = (START_LON + i * rotation_per_frame) % 360
    proj = ccrs.Orthographic(central_lon, CENTER_LAT)
    t = time_indices[i]
    ax = fig.add_subplot(1, 1, 1, projection=proj)
    im = ax.pcolormesh(
        lon, lat, rel_err[t],
        cmap="RdBu_r",
        vmin=-ERR_CLAMP, vmax=ERR_CLAMP,
        shading="auto", transform=pc,
    )
    ax.set_global()
    ax.gridlines(alpha=0.3)
    ax.set_title(f"Relative Error  t={t}")
    fig.colorbar(im, ax=ax, label="rel. error", shrink=0.8, pad=0.05)
    fig.tight_layout()
    return []

ani_err = animation.FuncAnimation(fig, make_frame_err, frames=N_FRAMES, blit=False)
plt.close()
HTML(ani_err.to_jshtml(fps=FPS))